In [6]:
# General imports
import pandas as pd
import os, warnings, datetime

from sklearn.preprocessing import LabelEncoder
import numpy as np
import pandas as pd
import sys, gc, random, datetime, math, psutil

import seaborn as sns
import matplotlib.pyplot as plt

from multiprocessing import Pool

warnings.filterwarnings('ignore')

In [7]:
########################### Helpers
#################################################################################
## -------------------
## Memory Reducer
# :df pandas dataframe to reduce size             # type: pd.DataFrame()
# :verbose                                        # type: bool

# minification the data usage
def reduce_mem_usage(df, verbose=True):
    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    start_mem = df.memory_usage().sum() / 1024 ** 2
    for col in df.columns:
        col_type = df[col].dtypes
        if col_type in numerics:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
    end_mem = df.memory_usage().sum() / 1024 ** 2
    if verbose: print('Mem. usage decreased to {:5.2f} Mb ({:.1f}% reduction)'.format(end_mem, 100 * (
            start_mem - end_mem) / start_mem))
    return df

## -------------------

In [8]:
########################### Vars
#################################################################################
START_DATE = datetime.datetime.strptime('2017-11-30', '%Y-%m-%d')

In [9]:
########################### DATA LOAD
#################################################################################
print('Load Data')
train_df = pd.read_csv('/ieee-fraud-detection/train_transaction.csv')
test_df = pd.read_csv('/ieee-fraud-detection/test_transaction.csv')
test_df['isFraud'] = 0

train_identity = pd.read_csv('/ieee-fraud-detection/train_identity.csv')
test_identity = pd.read_csv('/ieee-fraud-detection/test_identity.csv')

Load Data


In [10]:
########################### Base check
#################################################################################
from pandas.api.types import is_string_dtype

for df in [train_df, test_df, train_identity, test_identity]:
    original = df.copy()
    df = reduce_mem_usage(df)

    for col in list(df):
        if df[col].dtype != 'O':  # ignore string/object datatype, focus on numeric data.
            if not is_string_dtype(df[col]):  # make sure that the column datatype is not string
                if (df[col] - original[col]).sum() != 0:  # if col - org[col] != 0 => something wrongs.
                    df[col] = original[col]  # force it to become the original col.
                    print('Bad transformation', col)

Mem. usage decreased to 542.35 Mb (69.4% reduction)
Bad transformation TransactionAmt
Bad transformation dist1
Bad transformation dist2
Bad transformation C1
Bad transformation C2
Bad transformation C4
Bad transformation C6
Bad transformation C7
Bad transformation C8
Bad transformation C10
Bad transformation C11
Bad transformation C12
Bad transformation C13
Bad transformation D8
Bad transformation D9
Bad transformation V126
Bad transformation V127
Bad transformation V128
Bad transformation V129
Bad transformation V130
Bad transformation V131
Bad transformation V132
Bad transformation V133
Bad transformation V134
Bad transformation V135
Bad transformation V136
Bad transformation V137
Bad transformation V150
Bad transformation V159
Bad transformation V164
Bad transformation V202
Bad transformation V203
Bad transformation V204
Bad transformation V205
Bad transformation V206
Bad transformation V207
Bad transformation V208
Bad transformation V209
Bad transformation V210
Bad transformation V

In [11]:
########################### TransactionDT
from pandas.tseries.holiday import USFederalHolidayCalendar as calendar
import math
import datetime
import numpy as np

dates_range = pd.date_range(start='2017-10-01', end='2019-01-01')
us_holidays = calendar().holidays(start=dates_range.min(), end=dates_range.max())

for df in [train_df, test_df]:
    # Temporary variables for aggregation
    df['DT'] = df['TransactionDT'].apply(lambda x: (START_DATE + datetime.timedelta(seconds=x)))
    df['DT_M'] = ((df['DT'].dt.year - 2017) * 12 + df['DT'].dt.month).astype(np.int8)
    df['DT_W'] = ((df['DT'].dt.year - 2017) * 52 + df['DT'].dt.isocalendar().week).astype(
        np.int8)  # weekofyear is removed
    df['DT_D'] = ((df['DT'].dt.year - 2017) * 365 + df['DT'].dt.day_of_year).astype(np.int16)  # dayofyear → day_of_year

    df['DT_hour'] = (df['DT'].dt.hour).astype(np.int8)
    df['DT_day_week'] = (df['DT'].dt.day_of_week).astype(np.int8)  # dayofweek → day_of_week
    df['DT_day_month'] = (df['DT'].dt.day).astype(np.int8)
    df['DT_week_month'] = df['DT'].dt.day / 7
    df['DT_week_month'] = df['DT_week_month'].apply(math.ceil).astype(np.int8)

    # Possible solo feature
    df['is_december'] = (df['DT'].dt.month == 12).astype(np.int8)  # simplified, removed redundant assignment

    # Holidays
    df['is_holiday'] = (df['DT'].dt.normalize().isin(us_holidays)).astype(
        np.int8)  # replaced .dt.date.astype('datetime64')

# Total transactions per timeblock
for col in ['DT_M', 'DT_W', 'DT_D']:
    temp_df = pd.concat([train_df[[col]], test_df[[col]]])  # Combine the same column in train and test dataframe
    fq_encode = temp_df[col].value_counts().to_dict()  # Count how many transactions occurred each time block

    # Map them to train and test dataframe
    train_df[col + '_total'] = train_df[col].map(fq_encode)
    test_df[col + '_total'] = test_df[col].map(fq_encode)

In [12]:
########################### card4, card6, ProductCD
#################################################################################
# Converting Strings to ints(or floats if nan in column) using frequency encoding
# We will be able to use these columns as category or as numerical feature

for col in ['card4', 'card6', 'ProductCD']:
    print('Encoding', col)
    temp_df = pd.concat([train_df[[col]], test_df[[col]]])
    col_encoded = temp_df[col].value_counts().to_dict()  # count times that the values occurred.

    # Map them to train and test dataframe.
    train_df[col] = train_df[col].map(col_encoded)
    test_df[col] = test_df[col].map(col_encoded)
    print(col_encoded)

Encoding card4
{'visa': 719649, 'mastercard': 347386, 'american express': 16009, 'discover': 9524}
Encoding card6
{'debit': 824959, 'credit': 267648, 'debit or credit': 30, 'charge card': 16}
Encoding ProductCD
{'W': 800657, 'C': 137785, 'R': 73346, 'H': 62397, 'S': 23046}


In [13]:
########################### M columns
#################################################################################
# Converting Strings to ints(or floats if nan in column)

for col in ['M1', 'M2', 'M3', 'M5', 'M6', 'M7', 'M8', 'M9']:
    train_df[col] = train_df[col].map({'T': 1, 'F': 0})
    test_df[col] = test_df[col].map({'T': 1, 'F': 0})

for col in ['M4']:  # used frequency encoding.
    print('Encoding', col)
    temp_df = pd.concat([train_df[[col]], test_df[[col]]])
    col_encoded = temp_df[col].value_counts().to_dict()
    train_df[col] = train_df[col].map(col_encoded)
    test_df[col] = test_df[col].map(col_encoded)
    print(col_encoded)

Encoding M4
{'M0': 357789, 'M2': 122947, 'M1': 97306}


In [14]:
########################### Identity columns
#################################################################################
test_identity.columns = test_identity.columns.str.replace('-', '_', regex=False)
train_identity.columns = train_identity.columns.str.replace('-', '_', regex=False)


def minify_identity_df(df):
    df['id_12'] = df['id_12'].map({'Found': 1, 'NotFound': 0})
    df['id_15'] = df['id_15'].map({'New': 2, 'Found': 1, 'Unknown': 0})
    df['id_16'] = df['id_16'].map({'Found': 1, 'NotFound': 0})
    df['id_23'] = df['id_23'].map({'TRANSPARENT': 4, 'IP_PROXY': 3, 'IP_PROXY:ANONYMOUS': 2, 'IP_PROXY:HIDDEN': 1})
    df['id_27'] = df['id_27'].map({'Found': 1, 'NotFound': 0})
    df['id_28'] = df['id_28'].map({'New': 2, 'Found': 1})
    df['id_29'] = df['id_29'].map({'Found': 1, 'NotFound': 0})
    df['id_35'] = df['id_35'].map({'T': 1, 'F': 0})
    df['id_36'] = df['id_36'].map({'T': 1, 'F': 0})
    df['id_37'] = df['id_37'].map({'T': 1, 'F': 0})
    df['id_38'] = df['id_38'].map({'T': 1, 'F': 0})

    df['id_34'] = df['id_34'].fillna(':0').astype(str)
    df['id_34'] = df['id_34'].apply(lambda x: x.split(':')[1] if ':' in x else '0').astype(np.int8)
    df['id_34'] = np.where(df['id_34'] == 0, np.nan, df['id_34'])

    df['id_33'] = df['id_33'].fillna('0x0').astype(str)
    df['id_33_0'] = df['id_33'].apply(lambda x: x.split('x')[0] if 'x' in x else '0').astype(int)
    df['id_33_1'] = df['id_33'].apply(lambda x: x.split('x')[1] if 'x' in x else '0').astype(int)
    df['id_33'] = np.where(df['id_33'] == '0x0', np.nan, df['id_33'])

    df['DeviceType'] = df['DeviceType'].map({'desktop': 1, 'mobile': 0})
    return df


train_identity = minify_identity_df(train_identity)
test_identity = minify_identity_df(test_identity)

for col in ['id_33']:
    train_identity[col] = train_identity[col].fillna('unseen_before_label')
    test_identity[col] = test_identity[col].fillna('unseen_before_label')

    le = LabelEncoder()  # convert categorical data to numeric cal data from (0 to n)
    le.fit(list(train_identity[col]) + list(test_identity[col]))
    train_identity[col] = le.transform(train_identity[col])
    test_identity[col] = le.transform(test_identity[col])

In [15]:
########################### Deltas
for df in [train_df, test_df]:
    for col in ['D' + str(i) for i in range(1, 16) if
                i != 9]:  # we are not using the D9, as it represents the current hour in a day.
        new_col = 'uid_td_' + str(col)
        df[new_col] = df[col].fillna(0).astype(int)
        df[new_col] = df[new_col].apply(
            lambda x: pd.Timedelta(x, unit='D'))  # convert x int values, to Day: 5 -> 5 Days
        df[new_col] = (df['DT'] - df[new_col]).dt.date  # how long it was from the new_col's day to DT.
        df[new_col] = df[new_col].astype(str)
        df[new_col] = np.where(df[col].isna(), np.nan,
                               df[new_col])  # add a new boolean value column where df[col].isna()

In [16]:
########################### Final check##########################################
#################################################################################
# We have to reduce the memory usage again, as we have added new columns.
for df in [train_df, test_df, train_identity, test_identity]:
    original = df.copy()
    df = reduce_mem_usage(df)

    for col in list(df):
        if df[col].dtype != 'O':
            if not is_string_dtype(df[col]):
                if (df[col] - original[col]).sum() != 0:
                    df[col] = original[col]
                    print('Bad transformation', col)

Mem. usage decreased to 585.15 Mb (32.1% reduction)
Bad transformation TransactionAmt
Bad transformation dist1
Bad transformation dist2
Bad transformation C1
Bad transformation C2
Bad transformation C4
Bad transformation C6
Bad transformation C7
Bad transformation C8
Bad transformation C10
Bad transformation C11
Bad transformation C12
Bad transformation C13
Bad transformation D8
Bad transformation D9
Bad transformation V126
Bad transformation V127
Bad transformation V128
Bad transformation V129
Bad transformation V130
Bad transformation V131
Bad transformation V132
Bad transformation V133
Bad transformation V134
Bad transformation V135
Bad transformation V136
Bad transformation V137
Bad transformation V150
Bad transformation V159
Bad transformation V164
Bad transformation V202
Bad transformation V203
Bad transformation V204
Bad transformation V205
Bad transformation V206
Bad transformation V207
Bad transformation V208
Bad transformation V209
Bad transformation V210
Bad transformation V

In [17]:
########################### Export
#################################################################################
import pickle

train_df.to_pickle('train_transaction.pkl')
test_df.to_pickle('test_transaction.pkl')

train_identity.to_pickle('train_identity.pkl')
test_identity.to_pickle('test_identity.pkl')

In [18]:
###########################Full minification for fast tests
#################################################################################
for df in [train_df, test_df, train_identity, test_identity]:
    df = reduce_mem_usage(df)

Mem. usage decreased to 585.15 Mb (28.5% reduction)
Mem. usage decreased to 509.80 Mb (27.5% reduction)
Mem. usage decreased to 14.72 Mb (5.3% reduction)
Mem. usage decreased to 14.48 Mb (5.3% reduction)


In [19]:
########################### Export
#################################################################################
# just to make sure that the usage reduction is 0.0%
train_df.to_pickle('train_transaction_mini.pkl')
test_df.to_pickle('test_transaction_mini.pkl')

train_identity.to_pickle('train_identity_mini.pkl')
test_identity.to_pickle('test_identity_mini.pkl')

In [20]:
########################### Export
#################################################################################

possible_goups = [['V1', 'V2', 'V6', 'V7', 'V8', 'V9'],
                  ['V1', 'V2', 'V3', 'V6', 'V7', 'V8', 'V9'],
                  ['V2', 'V3', 'V6', 'V7', 'V8', 'V9'],
                  ['V4', 'V5'],
                  ['V10', 'V11'],
                  ['V12', 'V13'],
                  ['V14', 'V65'],
                  ['V15', 'V16', 'V33', 'V34', 'V57', 'V58', 'V79', 'V94'],
                  ['V15', 'V16', 'V33', 'V34', 'V57'],
                  ['V17', 'V18', 'V21', 'V22'],
                  ['V19', 'V20'],
                  ['V17', 'V18', 'V21', 'V22', 'V63', 'V84'],
                  ['V23', 'V24'],
                  ['V25', 'V26'],
                  ['V27', 'V28', 'V68', 'V89'],
                  ['V29', 'V30', 'V69', 'V90', 'V91'],
                  ['V29', 'V30', 'V70', 'V90', 'V91'],
                  ['V31', 'V32', 'V50', 'V71', 'V92', 'V93'],
                  ['V31', 'V32', 'V92'],
                  ['V15', 'V16', 'V33', 'V34', 'V51', 'V94'],
                  ['V15', 'V16', 'V33', 'V34'],
                  ['V35', 'V36'],
                  ['V37', 'V38'],
                  ['V39', 'V40'],
                  ['V41', 'V46', 'V47'],
                  ['V42', 'V43', 'V84'],
                  ['V42', 'V43'],
                  ['V44', 'V45'],
                  ['V48', 'V49'],
                  ['V31', 'V50', 'V71', 'V92'],
                  ['V33', 'V51', 'V52', 'V73', 'V94'],
                  ['V51', 'V52'],
                  ['V53', 'V54'],
                  ['V15', 'V16', 'V57', 'V58', 'V73', 'V79', 'V94'],
                  ['V15', 'V57', 'V58', 'V79'],
                  ['V59', 'V60', 'V63'],
                  ['V59', 'V60'],
                  ['V61', 'V62'],
                  ['V21', 'V59', 'V63', 'V64', 'V84'],
                  ['V63', 'V64'],
                  ['V66', 'V67'],
                  ['V29', 'V69', 'V70', 'V90'],
                  ['V30', 'V69', 'V70', 'V90', 'V91'],
                  ['V31', 'V50', 'V71', 'V72', 'V92', 'V93'],
                  ['V71', 'V72', 'V92', 'V93'],
                  ['V51', 'V57', 'V73', 'V74', 'V94'],
                  ['V73', 'V74'],
                  ['V75', 'V76'],
                  ['V80', 'V81', 'V84'],
                  ['V80', 'V81'],
                  ['V82', 'V83'],
                  ['V21', 'V42', 'V63', 'V80', 'V84', 'V85'],
                  ['V84', 'V85'],
                  ['V86', 'V87'],
                  ['V29', 'V30', 'V69', 'V70', 'V90', 'V91'],
                  ['V31', 'V32', 'V50', 'V71', 'V72', 'V92', 'V93'],
                  ['V31', 'V71', 'V72', 'V92', 'V93'],
                  ['V15', 'V33', 'V51', 'V57', 'V73', 'V94'],
                  ['V101',
                   'V102',
                   'V103',
                   'V143',
                   'V167',
                   'V168',
                   'V177',
                   'V178',
                   'V179',
                   'V279',
                   'V280',
                   'V293',
                   'V295',
                   'V322',
                   'V323',
                   'V324',
                   'V95',
                   'V96',
                   'V97'],
                  ['V101',
                   'V102',
                   'V103',
                   'V143',
                   'V167',
                   'V168',
                   'V177',
                   'V178',
                   'V179',
                   'V279',
                   'V280',
                   'V293',
                   'V294',
                   'V295',
                   'V322',
                   'V323',
                   'V324',
                   'V95',
                   'V96',
                   'V97'],
                  ['V105', 'V106', 'V296', 'V298', 'V299', 'V329', 'V330'],
                  ['V105', 'V106', 'V298', 'V299', 'V329', 'V330'],
                  ['V111', 'V113'],
                  ['V126', 'V128', 'V132', 'V134'],
                  ['V127', 'V128', 'V133', 'V134'],
                  ['V126', 'V127', 'V128', 'V132', 'V133', 'V134', 'V332'],
                  ['V129', 'V266', 'V269', 'V309', 'V334'],
                  ['V130', 'V310'],
                  ['V131', 'V312'],
                  ['V126', 'V128', 'V132', 'V133', 'V134'],
                  ['V127', 'V128', 'V132', 'V133', 'V134'],
                  ['V126', 'V127', 'V128', 'V132', 'V133', 'V134', 'V318', 'V332'],
                  ['V136', 'V137'],
                  ['V101',
                   'V102',
                   'V103',
                   'V143',
                   'V167',
                   'V177',
                   'V178',
                   'V179',
                   'V279',
                   'V280',
                   'V293',
                   'V295',
                   'V322',
                   'V323',
                   'V324',
                   'V95',
                   'V96',
                   'V97'],
                  ['V144', 'V145', 'V150', 'V151'],
                  ['V148', 'V149', 'V153', 'V154', 'V155', 'V156', 'V157', 'V158'],
                  ['V144', 'V145', 'V150', 'V151', 'V152'],
                  ['V151', 'V152'],
                  ['V161', 'V163'],
                  ['V162', 'V163'],
                  ['V161', 'V162', 'V163'],
                  ['V101',
                   'V102',
                   'V103',
                   'V167',
                   'V168',
                   'V177',
                   'V178',
                   'V179',
                   'V279',
                   'V280',
                   'V293',
                   'V295',
                   'V322',
                   'V323',
                   'V324',
                   'V95',
                   'V96',
                   'V97'],
                  ['V176', 'V190', 'V199', 'V228', 'V246', 'V257'],
                  ['V180', 'V182', 'V183'],
                  ['V181', 'V328'],
                  ['V180', 'V182', 'V183', 'V330'],
                  ['V186', 'V191', 'V196'],
                  ['V187', 'V192'],
                  ['V187', 'V192', 'V193'],
                  ['V192', 'V193', 'V196'],
                  ['V194', 'V197'],
                  ['V195', 'V198'],
                  ['V186', 'V191', 'V193', 'V196'],
                  ['V202', 'V204', 'V211', 'V213'],
                  ['V203', 'V204', 'V212'],
                  ['V202', 'V203', 'V204', 'V213'],
                  ['V202', 'V211', 'V213'],
                  ['V203', 'V212', 'V213'],
                  ['V202', 'V204', 'V211', 'V212', 'V213'],
                  ['V214', 'V276', 'V337'],
                  ['V215', 'V216', 'V277', 'V278', 'V338', 'V339'],
                  ['V217', 'V219', 'V231', 'V233'],
                  ['V218', 'V219', 'V232', 'V233'],
                  ['V217', 'V218', 'V219', 'V231', 'V232', 'V233'],
                  ['V222', 'V230'],
                  ['V224', 'V225'],
                  ['V229', 'V230', 'V258'],
                  ['V222', 'V229', 'V230', 'V258'],
                  ['V236', 'V237'],
                  ['V238', 'V239'],
                  ['V240', 'V241', 'V247', 'V252', 'V260'],
                  ['V242', 'V244'],
                  ['V245', 'V259'],
                  ['V240', 'V241', 'V247', 'V249', 'V252'],
                  ['V248', 'V249', 'V254'],
                  ['V247', 'V248', 'V249', 'V252'],
                  ['V250', 'V251'],
                  ['V248', 'V254'],
                  ['V255', 'V256'],
                  ['V240', 'V241', 'V260'],
                  ['V263', 'V265', 'V273', 'V274', 'V275'],
                  ['V264', 'V265'],
                  ['V263', 'V264', 'V265'],
                  ['V129', 'V266', 'V269', 'V309', 'V334', 'V336'],
                  ['V268', 'V336'],
                  ['V270', 'V272'],
                  ['V263', 'V273', 'V274', 'V275'],
                  ['V291', 'V292'],
                  ['V102', 'V280', 'V294', 'V295', 'V323', 'V96'],
                  ['V105', 'V296', 'V298', 'V299', 'V329'],
                  ['V105', 'V106', 'V296', 'V298', 'V299', 'V329'],
                  ['V105', 'V106', 'V296', 'V298', 'V299', 'V330'],
                  ['V300', 'V301'],
                  ['V302', 'V304'],
                  ['V303', 'V304'],
                  ['V302', 'V303', 'V304'],
                  ['V306', 'V308', 'V316', 'V318'],
                  ['V307', 'V308', 'V317'],
                  ['V306', 'V307', 'V308', 'V318'],
                  ['V313', 'V315'],
                  ['V306', 'V316', 'V318'],
                  ['V307', 'V317', 'V318'],
                  ['V134', 'V306', 'V308', 'V316', 'V317', 'V318'],
                  ['V320', 'V321'],
                  ['V326', 'V327'],
                  ['V105', 'V106', 'V296', 'V298', 'V329', 'V330'],
                  ['V105', 'V106', 'V183', 'V299', 'V329', 'V330'],
                  ['V331', 'V332', 'V333'],
                  ['V128', 'V134', 'V331', 'V332', 'V333'],
                  ['V335', 'V336'],
                  ['V266', 'V268', 'V269', 'V334', 'V335', 'V336']]

with open('possible_goups.pickle', 'wb') as f:
    pickle.dump(possible_goups, f, pickle.HIGHEST_PROTOCOL)

In [21]:
########################### Helpers
#################################################################################
## Multiprocessing Run.
# :df - DataFrame to split                      # type: pandas DataFrame
# :func - Function to apply on each split       # type: python function
# This function is NOT 'bulletproof', be carefull and pass only correct types of variables.
def df_parallelize_run(df, func):
    num_partitions, num_cores = psutil.cpu_count(), psutil.cpu_count()  # number of partitions and cores
    df_split = np.array_split(df, num_partitions)
    pool = Pool(num_cores)
    df = pd.concat(pool.map(func, df_split))
    pool.close()
    pool.join()
    return df


def check_state():
    if LOCAL_TEST:
        bad_uids = full_df.groupby(['uid'])['isFraud'].agg(['nunique', 'count'])
        bad_uids = bad_uids[(bad_uids['nunique'] == 2)]
        print('Inconsistent groups', len(bad_uids))

    print('Cleaning done...')
    print('Total groups:', len(full_df['uid'].unique()),
          '| Total items:', len(full_df),
          '| Total fraud', full_df['isFraud'].sum())


########################### Sainity check 
def sanity_check_run(temp_df, verbose=False):
    temp_df = temp_df.copy()
    temp_df = temp_df.sort_values(by='TransactionID').reset_index(drop=True)
    bad_uids_groups = pd.DataFrame()

    """
    for col in ['C1','C2','C3','C4','C5','C6','C7','C8','C9','C10','C11','C12','C13','C14']:
        temp_df['sanity_check'] = temp_df.groupby(['uid'])[col].shift()
        temp_df['sanity_check'] = (temp_df[col]-temp_df['sanity_check']).fillna(0).clip(None,0)

        bad_uids = temp_df.groupby(['uid'])['sanity_check'].agg(['sum']).reset_index()
        bad_uids = bad_uids[bad_uids['sum']<0]
        bad_uids_groups = pd.concat([bad_uids_groups,bad_uids])
        if verbose: print(col, len(bad_uids), bad_uids['uid'].values[:2])
    """
    bad_uids = temp_df.groupby(['uid'])['V313'].agg(['nunique']).reset_index()
    bad_uids = bad_uids[(bad_uids['nunique'] > 2)]
    bad_uids_groups = pd.concat([bad_uids_groups, bad_uids])
    if verbose: print('V313:', len(bad_uids), bad_uids['uid'].values[:2])

    bad_uids_groups = bad_uids_groups[['uid']].drop_duplicates()
    if verbose: print('Total bad groups:', len(bad_uids_groups))
    return bad_uids_groups


def parallel_check(bad_uids_groups):
    bad_uids_items = []
    if True:
        for cur_uid in list(bad_uids_groups['uid'].unique()):
            temp_df = full_df[full_df['uid'] == cur_uid].reset_index(drop=True)
            v313_values = temp_df['V313'].value_counts()
            if len(v313_values) > 1:
                v313_values = [[col for col in list(v313_values.index)[:2] if col != 0][0]] + [0]
            else:
                v313_values = [list(v313_values.index)[0]] + [0]

            for i in range(1, len(temp_df)):
                item_1 = temp_df.iloc[i]
                item_2 = temp_df.iloc[i - 1]

                check_if_match = temp_df.drop([i])
                check_if_match = sanity_check_run(check_if_match)
                if len(check_if_match) == 0:
                    bad_uids_items.append(item_1['TransactionID'])
                    break

                if i != 1:
                    check_if_match = temp_df.drop([i - 1])
                    check_if_match = sanity_check_run(check_if_match)
                    if len(check_if_match) == 0:
                        bad_uids_items.append(item_2['TransactionID'])
                        break

                """
                for col in ['C1','C2','C3','C4','C5','C6','C7','C8','C9','C10','C11','C12','C13','C14']:
                    check_sanity = item_1[col]<item_2[col]
                    if check_sanity:
                        bad_uids_items.append(item_1['TransactionID'])
                        break 
                        
                if check_sanity:
                    break
                """
                check_sanity = item_1['V313'] not in v313_values
                if check_sanity:
                    bad_uids_items.append(item_1['TransactionID'])
                    break

                if temp_df['TransactionID'].isin(problem_items['TransactionID']).sum() == 0:
                    if cur_uid >= 0:
                        check_sanity = ((item_2['DT_day'] - item_1['uid_td_D3']) ** 2) ** 0.5 > 1

                        if check_sanity:
                            bad_uids_items.append(item_1['TransactionID'])
                            break

    bad_uids_items = pd.DataFrame(bad_uids_items, columns=['uid'])
    return bad_uids_items


In [22]:
# Constants
LOCAL_TEST = False # or False
CHECK_ORDER = True
TRUST_D1 = True
MULTI_UID_CHECK = False
FULL_GROUP_CHECK = False

In [23]:
print('Load Data')
train_df = pd.read_pickle('train_transaction.pkl')
test_df = pd.read_pickle('test_transaction.pkl')

full_df = pd.concat([train_df, test_df]).reset_index(drop=True)

if LOCAL_TEST:
    full_df = full_df.iloc[:10000]  #full_df[(full_df['DT_M']==12)]

Load Data


In [24]:
########################### Base preparation
# 
full_df['full_addr'] = full_df['addr1'].astype(str) + '_' + full_df['addr2'].astype(str)

for col in ['D' + str(i) for i in [1, 2, 3, 5, 10, 11, 15]]:
    new_col = 'uid_td_' + str(col)
    full_df[new_col] = full_df['TransactionDT'] / (24 * 60 * 60)  # convert to day
    full_df[new_col] = np.floor(full_df[new_col] - full_df[col]) + 1000  #offset: used to prevent negative delta values.

full_df['DT_day'] = np.floor(full_df['TransactionDT'] / (24 * 60 * 60)) + 1000  # create a day, like: DT_day = 1.

full_df['TransactionAmt_fix'] = np.round(full_df['TransactionAmt'], 2)
full_df['V313_fix'] = np.round(full_df['V313'], 2)
full_df['uid'] = np.nan

v_cols = []
v_fix_cols = []
for col in ['V' + str(i) for i in range(1, 340)]:
    if (full_df[col].fillna(0) - full_df[col].fillna(0).astype(
            int)).sum() != 0:  # check if the null values converted to 0.
        if col not in ['V313']:  # Except V313
            v_cols.append(col)
            v_fix_cols.append(col + '_fix')
            full_df[col + '_fix_ground'] = np.round(full_df[col], 2)  #round to second digits
            full_df[col + '_fix'] = full_df[col + '_fix_ground'] + full_df['TransactionAmt_fix']

global_bad_items = full_df[full_df['D1'].isna()]  # store the rows where D1s are null

# save the dataframe exclude TransactionDT, where rows are not belong to global_bad_items.
full_df = full_df[~full_df['TransactionDT'].isin(global_bad_items['TransactionDT'])]
all_items = full_df.copy()
bkp_items = full_df.copy()  # back up dataframe

print('Total number of transactions:', len(full_df))

Total number of transactions: 1089720


In [25]:
 ########################### Single Transaction
# Let's filter single card appearance card1/D1 -> single transaction per

# group transactions that have the same card1 and uid_td_D1, and look for unique ID, and add to 'count' column
full_df['count'] = full_df.groupby(['card1', 'uid_td_D1'])['TransactionID'].transform('count')
single_items = full_df[full_df['count'] == 1]
single_items['uid'] = single_items['TransactionID']
del full_df, single_items['count']

all_items = all_items[~all_items['TransactionID'].isin(single_items['TransactionID'])] #
print('Single transaction', len(single_items))

Single transaction 125242


In [26]:
### Clean full_df
full_df = pd.DataFrame()

In [27]:
# Create a unique uid for every person for the very first time.

first_df = all_items.copy()

first_df['counts'] = first_df.groupby(['card1', 'uid_td_D1']).cumcount()  # how many transactions in the sequence.
first_df = first_df[first_df['counts'] == 0]
del first_df['counts']  # delete the first_df where counts = 0.

first_df['uid'] = first_df['TransactionID']  #set the uid = transaction id
print('First time in dataset', len(first_df))

full_df = pd.concat([full_df, first_df])  #
full_df = full_df.sort_values(by='TransactionID').reset_index(drop=True)
del first_df

First time in dataset 137840


In [28]:
check_state()

Cleaning done...
Total groups: 137840 | Total items: 137840 | Total fraud 2621


In [29]:
# Let's Check unassigned items again.
# Let's find items with roots out of our dataset
nan_df_check = all_items[~all_items['TransactionID'].isin(full_df['TransactionID'])]

# if 'uid_td_D3'>1000 it means that root item is in our dataset
# >1000 will also filter NaNs values
nan_df_check['uid'] = np.where(nan_df_check['uid_td_D3'] >= 1001,
                               np.nan, nan_df_check['TransactionID'])
nan_df_check = nan_df_check[~nan_df_check['uid'].isna()]  # remove the nan rows (>=1001)

full_df = pd.concat([full_df, nan_df_check])
full_df = full_df.sort_values(by='TransactionID').reset_index(drop=True)

print('Roots out of dataset', len(nan_df_check))
#del nan_df_check
out_of_bonds = nan_df_check[['TransactionID']]

Roots out of dataset 261173


In [30]:
check_state()

Cleaning done...
Total groups: 399013 | Total items: 399013 | Total fraud 9744


In [31]:
########################### VERY IMPORTANT
# Do not do sanity D3 check for gap items
# This means there's roughly a 30 day gap in the dataset where no transactions exist. This is a known property of the IEEE-CIS dataset — Vesta simply had no data for that period.
problem_items = full_df[(full_df['uid_td_D3'] > 1182) & (full_df['uid_td_D3'] < 1213)] # in range (1182, 1213)
out_of_bonds = pd.concat([out_of_bonds, problem_items[['TransactionID']]])

In [32]:
########################### Sort
all_items = all_items.sort_values(by='TransactionID').reset_index(drop=True)
single_items = single_items.sort_values(by='TransactionID').reset_index(drop=True)
full_df = full_df.sort_values(by='TransactionID').reset_index(drop=True)
out_of_bonds = out_of_bonds.sort_values(by='TransactionID').reset_index(drop=True)

In [33]:
def find_and_append_root(df: pd.DataFrame):
    new_uids_items = {'TransactionID': [],
                      'uid': [],
                      }

    for i in range(len(df)):
        item = df.iloc[i]  # just the row of dataframe
        if item['TransactionID'] not in list(
                problem_items['TransactionID']):  # the problem items are handled separately.
            mask_1 = bkp_items['card1'] == item['card1']  # list of the same card1 (true or false)
            mask_2 = bkp_items['uid_td_D1'] == item[
                'uid_td_D1']  # list of the same uid_td_D1: the days the previous transaction
            mask_3 = bkp_items['TransactionID'] < item[
                'TransactionID']  # the transactionID is monotonic and we are looking for the root.
            mask_4 = ((bkp_items['DT_day'] == item['uid_td_D3'] + 1) |
                      (bkp_items['DT_day'] == item['uid_td_D3'] - 1) |
                      (bkp_items['DT_day'] == item['uid_td_D3']))

            df_masked = bkp_items[mask_1 & mask_2 & mask_3 & mask_4]  # looking for rows that satisfies conditions
            no_match = len(df_masked) == 0 # true or false.

            if no_match:
                new_uids_items['TransactionID'].append(item['TransactionID'])
                new_uids_items['uid'].append(item['TransactionID'])

    return_df = pd.DataFrame.from_dict(new_uids_items)  # if the root is not found, this will become root.
    return return_df

In [34]:
########################### PART X - > 100% Root
import psutil
import numpy as np
from joblib import Parallel, delayed

def df_parallelize_run(df, func):
    num_cores = psutil.cpu_count()
    df_split = [pd.DataFrame(chunk, columns=df.columns)
                for chunk in np.array_split(df, num_cores)]  # convert back to DataFrame
    results = Parallel(n_jobs=num_cores)(
        delayed(func)(chunk) for chunk in df_split
    )
    return pd.concat(results).reset_index(drop=True)

nan_df = all_items[~all_items['TransactionID'].isin(full_df['TransactionID'])]
nan_df = nan_df[nan_df['DT_M'] < 18]
print('Items to check:', len(nan_df))

df_cleaned = df_parallelize_run(nan_df, find_and_append_root)
df_cleaned = df_cleaned[~df_cleaned['uid'].isna()]

df_cleaned.index = df_cleaned['TransactionID']
temp_dict = df_cleaned['uid'].to_dict()
nan_df['uid'] = nan_df['TransactionID'].map(temp_dict)
nan_df = nan_df[~nan_df['uid'].isna()]
print('Assigned root items:', len(nan_df))

# append found items
full_df = pd.concat([full_df, nan_df]).sort_values(by='TransactionID').reset_index(drop=True)
check_state()

Items to check: 283166


KeyboardInterrupt: 

In [ ]:
def append_item_to_uid(df):
    new_uids_items = {'TransactionID': [],
                      'uid': [],
                      }

    for i in range(len(df)):
        item = df.iloc[i]

        mask_1 = full_df['card1'] == item['card1']
        mask_2 = full_df['uid_td_D1'] == item['uid_td_D1']
        mask_3 = full_df['TransactionID'] < item['TransactionID']
        mask_4 = full_df['DT_day'] <= item['uid_td_D3'] + 1
        df_masked = full_df[mask_1 & mask_2 & mask_3 & mask_4]

        has_match = len(df_masked) > 0
        can_be_root = True

        for col in ['addr2', 'addr1']:
            if has_match:
                if not np.isnan(item[col]):
                    mask = ((df_masked[col] == item[col]) | (df_masked[col].isna()))
                    df_masked = df_masked[mask]
                if len(df_masked) == 0:
                    has_match = False

        if has_match:
            mask = (df_masked['TransactionID'] > item['TransactionID']).astype(int)
            for col in v_cols:
                mask += (df_masked[col + '_fix'] == item[col + '_fix_ground']).astype(int)
            mask = mask > 0
            df_masked = df_masked[mask]

            if len(df_masked) == 0:
                has_match = False

        if has_match and len(df_masked['uid'].unique()) == 1:
            # Fix: replace deprecated df_masked.append(item) with pd.concat
            check_if_match = pd.concat([df_masked, item.to_frame().T.astype(df_masked.dtypes)])
            check_if_match = sanity_check_run(check_if_match)
            if len(check_if_match) == 0:
                new_uids_items['TransactionID'].append(item['TransactionID'])
                new_uids_items['uid'].append(df_masked['uid'].unique()[0])

    return_df = pd.DataFrame.from_dict(new_uids_items)
    return return_df

In [ ]:
########################### PART X - > 100% single match
for outer_round in range(5):                          # fix: renamed i to outer_round to avoid conflict with inner loop
    print('Check round:', outer_round)

    nan_df = all_items[~all_items['TransactionID'].isin(full_df['TransactionID'])]
    print('Items to check:', len(nan_df))

    df_cleaned = df_parallelize_run(nan_df, append_item_to_uid)
    df_cleaned = df_cleaned[~df_cleaned['uid'].isna()]

    df_cleaned.index = df_cleaned['TransactionID']
    temp_dict = df_cleaned['uid'].to_dict()
    nan_df['uid'] = nan_df['TransactionID'].map(temp_dict)
    nan_df = nan_df[~nan_df['uid'].isna()]
    print('Assigned items:', len(nan_df))

    full_df = pd.concat([full_df, nan_df]).sort_values(by='TransactionID').reset_index(drop=True)

    for inner_round in range(100):                    # fix: renamed i to inner_round to avoid conflict with outer loop
        bad_uids_groups = sanity_check_run(full_df, False)
        if len(bad_uids_groups) == 0:
            break
        elif len(bad_uids_groups) > 64:
            bad_uids_items = df_parallelize_run(bad_uids_groups, parallel_check)
        else:
            bad_uids_items = parallel_check(bad_uids_groups)

        print('Found bad items', len(bad_uids_items))
        full_df['uid'] = np.where(full_df['TransactionID'].isin(bad_uids_items['uid']), np.nan, full_df['uid'])
        full_df = full_df[~full_df['uid'].isna()].sort_values(by='TransactionID').reset_index(drop=True)
        if len(bad_uids_items) < 2:
            break

    check_state()

In [ ]:
def find_and_append_root_test(df):
    new_uids_items = {'TransactionID': [],
                      'uid': [],
                      }

    for i in range(len(df)):
        item = df.iloc[i]
        if item['TransactionID'] not in list(problem_items['TransactionID']):
            mask_1 = bkp_items['card1'] == item['card1']
            mask_2 = bkp_items['uid_td_D1'] == item['uid_td_D1']
            mask_3 = bkp_items['TransactionID'] < item['TransactionID']
            mask_4 = ((bkp_items['DT_day'] == item['uid_td_D3'] + 1) |
                      (bkp_items['DT_day'] == item['uid_td_D3'] - 1) |
                      (bkp_items['DT_day'] == item['uid_td_D3']))

            df_masked = bkp_items[mask_1 & mask_2 & mask_3 & mask_4]
            no_match = len(df_masked) == 0

            if no_match:
                new_uids_items['TransactionID'].append(item['TransactionID'])
                new_uids_items['uid'].append(item['TransactionID'])

    return_df = pd.DataFrame.from_dict(new_uids_items)
    return return_df

In [ ]:
########################### PART X - > 100% Root

nan_df = all_items[~all_items['TransactionID'].isin(full_df['TransactionID'])]
nan_df = nan_df[nan_df['DT_M'] > 18]
print('Items to check:', len(nan_df))

df_cleaned = df_parallelize_run(nan_df, find_and_append_root_test)
df_cleaned = df_cleaned[~df_cleaned['uid'].isna()]
df_cleaned = df_cleaned[~df_cleaned['TransactionID'].isin(full_df['TransactionID'])]

df_cleaned.index = df_cleaned['TransactionID']
temp_dict = df_cleaned['uid'].to_dict()
nan_df['uid'] = nan_df['TransactionID'].map(temp_dict)
nan_df = nan_df[~nan_df['uid'].isna()]
print('Assigned root items:', len(nan_df))

# append found items
full_df = pd.concat([full_df, nan_df]).sort_values(by='TransactionID').reset_index(drop=True)
check_state()

In [ ]:
########################### PART X - > 100% single match
for i in range(3):
    print('Check round:', i)

    nan_df = all_items[~all_items['TransactionID'].isin(full_df['TransactionID'])]
    print('Items to check:', len(nan_df))

    df_cleaned = df_parallelize_run(nan_df, append_item_to_uid)
    df_cleaned = df_cleaned[~df_cleaned['uid'].isna()]

    df_cleaned.index = df_cleaned['TransactionID']
    temp_dict = df_cleaned['uid'].to_dict()
    nan_df['uid'] = nan_df['TransactionID'].map(temp_dict)
    nan_df = nan_df[~nan_df['uid'].isna()]
    print('Assigned items:', len(nan_df))

    # append found items
    full_df = pd.concat([full_df, nan_df]).sort_values(by='TransactionID').reset_index(drop=True)

    for i in range(100):
        bad_uids_groups = sanity_check_run(full_df, False)
        if len(bad_uids_groups) == 0:
            break
        elif len(bad_uids_groups) > 64:
            bad_uids_items = df_parallelize_run(bad_uids_groups, parallel_check)
        else:
            bad_uids_items = parallel_check(bad_uids_groups)

        print('Found bad items', len(bad_uids_items))
        full_df['uid'] = np.where(full_df['TransactionID'].isin(bad_uids_items['uid']), np.nan, full_df['uid'])
        full_df = full_df[~full_df['uid'].isna()].sort_values(by='TransactionID').reset_index(drop=True)
        if len(bad_uids_items) < 2:
            break
    check_state()

In [ ]:
full_df['isFraud'].value_counts()

In [ ]:
full_df[['TransactionID', 'uid']].to_csv('uids_part_1_v6.csv')

In [ ]:
def find_multigroup(df):
    new_uids_items = {'TransactionID': [],
                      'multi_uid': [],
                      }

    for i in range(len(df)):
        item = df.iloc[i]
        if item['TransactionID'] not in problem_items['TransactionID']:
            mask_1 = full_df['card1'] == item['card1']
            mask_2 = full_df['uid_td_D1'] == item['uid_td_D1']
            mask_3 = full_df['TransactionID'] < item['TransactionID']
            mask_4 = ((full_df['DT_day'] == item['uid_td_D3'] + 1) |
                      (full_df['DT_day'] == item['uid_td_D3'] - 1) |
                      (full_df['DT_day'] == item['uid_td_D3']))

            df_masked = full_df[mask_1 & mask_2 & mask_3 & mask_4]
            has_match = len(df_masked) > 0

            if has_match:
                new_uids_items['TransactionID'].append(item['TransactionID'])
                new_uids_items['multi_uid'].append(list(df_masked['uid'].unique()))

    return_df = pd.DataFrame.from_dict(new_uids_items)
    return return_df


def find_and_filter_groups(df):
    filtered_groups = []

    for i in range(len(df)):
        test_id = df.iloc[i]['TransactionID']
        test_item = all_items[all_items['TransactionID'] == test_id].iloc[0]
        possible_groups = df.iloc[i]['multi_uid']
        clean_group = find_right_uid(possible_groups, test_item)
        filtered_groups.append([test_id, clean_group])
    filtered_groups = pd.DataFrame(filtered_groups, columns=['TransactionID', 'uid'])
    return filtered_groups


import operator


def find_right_uid(possible_groups, test_item):
    separated_uids = {}

    test_features_set1 = {
        'TransactionAmt': 2,
        'card2': 1,
        'card3': 1,
        'card4': 1,
        'card5': 1,
        'card6': 1,
        'uid_td_D2': 2,
        'uid_td_D10': 2,
        'uid_td_D11': 2,
        'uid_td_D15': 2,
        'C14': 1,
        'addr1': 1,
        'addr2': 1,
        'P_emaildomain': 1,
        'V313_fix': 1,
    }

    groups_score = {}

    for possible_group in possible_groups:
        masked_df = full_df[full_df['uid'] == possible_group]
        cur_score = 0
        for col in test_features_set1:
            if test_item[col] in list(masked_df[col]):
                cur_score += test_features_set1[col]

        for col in v_cols:
            if test_item[col] != 0:
                if test_item[col + '_fix_ground'] in list(masked_df[col + '_fix']):
                    cur_score += 1

        check_if_match = masked_df.append(test_item)
        check_if_match = sanity_check_run(check_if_match)
        if len(check_if_match) == 0:
            groups_score[possible_group] = cur_score

    new_uid = np.nan
    try:
        new_uid = max(groups_score.items(), key=operator.itemgetter(1))[0]
    except:
        pass
    return new_uid

In [ ]:
########################### PART X - > With multigroup check
nan_df = all_items[~all_items['TransactionID'].isin(full_df['TransactionID'])]
print('Items to check:', len(nan_df))

df_cleaned = df_parallelize_run(nan_df, find_multigroup)
df_cleaned = df_cleaned[~df_cleaned['multi_uid'].isna()]

filtered_groups = df_parallelize_run(df_cleaned, find_and_filter_groups)
filtered_groups.index = filtered_groups['TransactionID']
temp_dict = filtered_groups['uid'].to_dict()
nan_df['uid'] = nan_df['TransactionID'].map(temp_dict)
nan_df = nan_df[~nan_df['uid'].isna()]
print('Assigned items:', len(nan_df))

# append found items
full_df = pd.concat([full_df, nan_df]).sort_values(by='TransactionID').reset_index(drop=True)
check_state()


In [ ]:
for i in range(100):
    bad_uids_groups = sanity_check_run(full_df, False)
    if len(bad_uids_groups) == 0:
        break
    elif len(bad_uids_groups) > 64:
        bad_uids_items = df_parallelize_run(bad_uids_groups, parallel_check)
    else:
        bad_uids_items = parallel_check(bad_uids_groups)

    print('Found bad items', len(bad_uids_items))
    full_df['uid'] = np.where(full_df['TransactionID'].isin(bad_uids_items['uid']), np.nan, full_df['uid'])
    full_df = full_df[~full_df['uid'].isna()].sort_values(by='TransactionID').reset_index(drop=True)
    if len(bad_uids_items) < 2:
        break
check_state()

In [ ]:
print('Start items:', len(bkp_items))
print('Start items Frauds:', bkp_items['isFraud'].sum())

In [ ]:
print('Uids Items:', len(single_items) + len(full_df))
print('Uids Frauds:', full_df['isFraud'].sum() + single_items['isFraud'].sum())

In [ ]:
full_df_final = pd.concat([full_df,
                           single_items,
                           global_bad_items,
                           all_items[~all_items['TransactionID'].isin(full_df['TransactionID'])]
                           ])

In [ ]:
print('Combined Uids:', len(full_df_final))
print('CombinedUids Frauds:', full_df_final['isFraud'].sum())

In [ ]:
full_df['count'] = full_df.groupby(['uid'])['TransactionID'].transform('count')
full_df['count'].mean()

In [ ]:
len(full_df_final['uid'].unique())

In [ ]:
check_state()

In [ ]:
full_df['isFraud']